# Model tests

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch import nn

In [3]:
from pathlib import Path
import json
import numpy as np
from tokenizers import Tokenizer

import torch
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
from src.gpt.model import RMSNorm, RoPE, Attention, MLP, Block, GPT

## RMS Norm

In [4]:
import torch
from torch import nn
import torch.nn.functional as F


d = 64
x = torch.randn(4, 10, d)
norm = RMSNorm(d)

# 1) independent reference: x / rms == normalize(x) * sqrt(d)
ref = F.normalize(x, dim=-1) * d ** 0.5
print(torch.allclose(norm(x), ref, atol=1e-4))

# 2) output RMS is ~1 at every position (gain = 1 at init)
print(norm(x).pow(2).mean(-1).sqrt())            # all ≈ 1.0

# 3) scale invariance
print(torch.allclose(norm(x), norm(x * 100), atol=1e-4))

# 4) the gain scales the output per channel
norm.weight.data.fill_(2.0)
print(torch.allclose(norm(x), 2 * ref, atol=1e-4))

True
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000]], grad_fn=<SqrtBackward0>)
True
True


## RoPE

In [5]:
rope = RoPE(head_dim=16, max_len=64)

# 1) rotation preserves norms
x = torch.randn(2, 3, 10, 16)
print(torch.allclose(x.norm(dim=-1), rope(x).norm(dim=-1), atol=1e-5))

# 2) position 0 is the identity (cos=1, sin=0)
print(torch.allclose(rope(x)[:, :, 0], x[:, :, 0], atol=1e-6))

# 3) relative position: same q and k vector at every position, so scores depend only on m - n
q = torch.randn(1, 1, 1, 16).expand(1, 1, 32, 16)
k = torch.randn(1, 1, 1, 16).expand(1, 1, 32, 16)
S = rope(q)[0, 0] @ rope(k)[0, 0].T                  # [32, 32]
print(all(torch.allclose(S.diagonal(o), S.diagonal(o)[0].expand_as(S.diagonal(o)), atol=1e-5)
          for o in range(-10, 11)))

# 4) start_pos matches slicing a full pass (a preview of the KV-cache test)
full = rope(x)
print(all(torch.allclose(rope(x[:, :, t:t+1], start_pos=t), full[:, :, t:t+1], atol=1e-6)
          for t in range(10)))

True
True
True
True


## Attention

In [6]:
attn = Attention(d_model=64, n_head=4, max_len=32)
x = torch.randn(2, 12, 64)

# 1) shape
print(attn(x).shape == x.shape)

# 2) causality: changing token t must not affect outputs before t
t = 7
x2 = x.clone()
x2[:, t] += torch.randn(2, 64)
y1, y2 = attn(x), attn(x2)
print(torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6))    # True: earlier positions unchanged
print((y1[:, t:] - y2[:, t:]).abs().max() > 1e-4)          # True: position t onward does change

True
True
tensor(True)


## GPT

In [7]:
from src.gpt.config import load_config
import math
import sys
import os
%cd ~/Desktop/swGPT
print(os.getcwd())

cfg = load_config("configs/tiny.yaml")
model = GPT(cfg)
idx = torch.randint(0, cfg.vocab_size, (4, cfg.context_len))
logits, loss = model(idx, idx)               # random targets are fine for this check

print(logits.shape == (4, cfg.context_len, cfg.vocab_size))
print(loss.item(), math.log(cfg.vocab_size))  # should be close, slightly above
print(sum(p.numel() for p in model.parameters()))

/Users/marzioformica/Desktop/swGPT
/Users/marzioformica/Desktop/swGPT


TypeError: Attention.__init__() takes 4 positional arguments but 5 were given